In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q torch_geometric
!pip install -q class_resolver
!pip3 install pymatting

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 2.6 MB/s eta 0:00:00


In [4]:
# -*- coding: utf-8 -*-
"""
PneumoniaMNIST + ResNet18 features + ARMA_Recon
Merged implementation: ResNet18 extraction from PneumoniaMNIST script,
ARMA_Recon model/training (ARMAConv encoder + reconstruction decoder + classifier).
"""

import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as nnFn
import torch.optim as optim

from torch_geometric.data import Data
from torch_geometric.nn import ARMAConv

from torch.utils.data import TensorDataset, DataLoader, Subset
from torchvision import models
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    log_loss,
)

# ============================================================
# Reproducibility
# ============================================================
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# ============================================================
# Load PneumoniaMNIST 224x224
# ============================================================
data = np.load(
    "/content/drive/MyDrive/TejaswiAbburi_va797/Dataset/Medmnist_data/pneumoniamnist_224.npz",
    allow_pickle=True,
)

all_images = np.concatenate(
    [data["train_images"], data["val_images"], data["test_images"]], axis=0
)
all_labels = np.concatenate(
    [data["train_labels"], data["val_labels"], data["test_labels"]], axis=0
).squeeze()

images = all_images.astype(np.float32) / 255.0
images = np.repeat(images[:, None, :, :], 3, axis=1)  # (N,3,224,224)

X = torch.tensor(images)
y = torch.tensor(all_labels).long()

print("Images, labels shapes:", X.shape, y.shape)

# ============================================================
# Balanced subset (same as PneumoniaMNIST ARMA script)
# ============================================================
dataset = TensorDataset(X, y)

class0_indices = [i for i in range(len(y)) if y[i] == 0]
class1_indices = [i for i in range(len(y)) if y[i] == 1]

random.seed(SEED)
sampled_class0 = random.sample(class0_indices, min(2000, len(class0_indices)))
sampled_class1 = random.sample(class1_indices, min(2000, len(class1_indices)))

combined_indices = sampled_class0 + sampled_class1
random.shuffle(combined_indices)

final_dataset = Subset(dataset, combined_indices)
final_loader = DataLoader(final_dataset, batch_size=64, shuffle=False)

# ============================================================
# Extract ResNet18 features
# ============================================================
resnet = models.resnet18(pretrained=True)
resnet.fc = nn.Identity()
resnet = resnet.to(device)
resnet.eval()

resnet_feats, y_list = [], []

with torch.no_grad():
    for imgs, lbls in final_loader:
        imgs = imgs.to(device)
        feats = resnet(imgs)
        resnet_feats.append(feats.cpu())
        y_list.extend(lbls.cpu().tolist())

features = torch.cat(resnet_feats, dim=0).numpy().astype(np.float32)
y_labels = np.array(y_list).astype(np.int64)

num_nodes, num_feats = features.shape
print(f"Extracted ResNet18 Features: {features.shape}, Labels: {y_labels.shape}")

# ============================================================
# Graph construction
# ============================================================
ALPHA = 0.9  # from PneumoniaMNIST ARMA script


def create_adj(F_data, alpha=0.9):
    norms = np.linalg.norm(F_data, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    F_norm = F_data / norms
    W = np.dot(F_norm, F_norm.T)
    W = np.where(W >= alpha, 1.0, 0.0).astype(np.float32)
    if W.max() > 0:
        W = W / W.max()
    return W


def load_data(adj, node_feats):
    node_feats = torch.from_numpy(node_feats).float()
    rows, cols = np.nonzero(adj > 0)
    edge_index = torch.from_numpy(np.vstack([rows, cols]).astype(np.int64))
    return Data(x=node_feats, edge_index=edge_index)


W = create_adj(features, alpha=ALPHA)
np.fill_diagonal(W, 1.0)

data_graph = load_data(W, features).to(device)
print(data_graph)

# ============================================================
# ARMA_Recon model
# ============================================================
class ARMA_AE(nn.Module):
    def __init__(
        self,
        input_dim,
        hidden_dim,
        output_dim,
        device,
        activ="SELU",
        dropout=0.25,
    ):
        super(ARMA_AE, self).__init__()
        self.device = device

        self.conv1 = ARMAConv(
            input_dim,
            hidden_dim,
            num_stacks=1,
            num_layers=1,
        )
        self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, output_dim)

        self.decoder = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim),
        )

        activations = {
            "SELU": nnFn.selu,
            "SiLU": nnFn.silu,
            "GELU": nnFn.gelu,
            "RELU": nnFn.relu,
            "ELU": nnFn.elu,
        }
        self.act = activations.get(activ, nnFn.relu)

    def forward(self, x, edge_index):
        h = self.conv1(x, edge_index)
        h = self.bn1(h)
        h = self.act(h)
        h = self.dropout(h)

        logits = self.fc(h)
        recon = self.decoder(h)
        return logits, recon, h


# ============================================================
# Hyperparameters (from PneumoniaMNIST ARMA script)
# ============================================================
feats_dim = num_feats
hidden_dim = 256
num_classes = 2
num_epochs = 2000
lr = 1e-4
weight_decay = 1e-4
batch_print_freq = 100
lambda_recon = 0.01   # ARMA_Recon default

# Outer/inner split (same protocol as the ARMA_PneumoniaMNIST script)
OUTER_TEST_SIZE = 0.9
INNER_TEST_SIZE = 0.9

# ============================================================
# Cross-validation
# ============================================================
sss = StratifiedShuffleSplit(
    n_splits=20,
    test_size=OUTER_TEST_SIZE,
    random_state=SEED,
)

accuracies, precisions, recalls, f1_scores, aucs, ce_losses = [], [], [], [], [], []

y_all = y_labels.copy()

for fold, (train_val_idx, test_idx_global) in enumerate(
    sss.split(features, y_all), start=1
):
    print(f"\n=== Fold {fold} ===")

    control_idx = np.where(y_all == 0)[0]
    patient_idx = np.where(y_all == 1)[0]

    sss_class = StratifiedShuffleSplit(
        n_splits=20,
        test_size=INNER_TEST_SIZE,
        random_state=fold,
    )

    control_train_idx, _ = next(
        sss_class.split(features[control_idx], y_all[control_idx])
    )
    patient_train_idx, _ = next(
        sss_class.split(features[patient_idx], y_all[patient_idx])
    )

    control_train = control_idx[control_train_idx]
    patient_train = patient_idx[patient_train_idx]

    train_idx_final = np.concatenate([control_train, patient_train])
    np.random.shuffle(train_idx_final)

    print(f"Train Class0: {len(control_train)}, Train Class1: {len(patient_train)}")

    train_idx_t = torch.from_numpy(train_idx_final).long().to(device)
    y_train_tensor = torch.from_numpy(y_all[train_idx_final]).long().to(device)

    model = ARMA_AE(
        feats_dim,
        hidden_dim,
        num_classes,
        device,
        activ="RELU",
        dropout=0.25,
    ).to(device)

    optimizer = optim.Adam(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay,
    )

    ce_loss_fn = nn.CrossEntropyLoss()
    recon_loss_fn = nn.MSELoss()

    for epoch in range(1, num_epochs + 1):
        model.train()
        optimizer.zero_grad()

        logits, recon_x, _ = model(data_graph.x, data_graph.edge_index)

        loss_sup = ce_loss_fn(logits[train_idx_t], y_train_tensor)
        loss_recon = recon_loss_fn(recon_x, data_graph.x)
        total_loss = loss_sup + lambda_recon * loss_recon

        if torch.isnan(total_loss) or torch.isinf(total_loss):
            print(
                f"NaN/Inf detected at fold {fold} epoch {epoch}. Aborting epoch."
            )
            break

        total_loss.backward()
        optimizer.step()

        if epoch % batch_print_freq == 0 or epoch == 1:
            model.eval()
            with torch.no_grad():
                logits_eval, _, _ = model(
                    data_graph.x, data_graph.edge_index
                )
                preds_train = logits_eval[train_idx_t].argmax(dim=1)
                acc_train = accuracy_score(
                    y_train_tensor.cpu(), preds_train.cpu()
                )

            print(
                f"Fold {fold} Epoch {epoch}: "
                f"TotalLoss={total_loss.item():.6f} | "
                f"Sup={loss_sup.item():.6f} | "
                f"Recon={loss_recon.item():.6f} | "
                f"TrainAcc={acc_train:.4f}"
            )

    # Evaluation
    model.eval()
    with torch.no_grad():
        out_logits, _, _ = model(data_graph.x, data_graph.edge_index)
        preds = out_logits.argmax(dim=1).cpu().numpy()
        probs = nnFn.softmax(out_logits, dim=1)[:, 1].cpu().numpy()

    y_test = y_all[test_idx_global]
    y_pred_test = preds[test_idx_global]
    y_prob_test = probs[test_idx_global]

    acc = accuracy_score(y_test, y_pred_test)
    prec = precision_score(y_test, y_pred_test, zero_division=0)
    rec = recall_score(y_test, y_pred_test, zero_division=0)
    f1 = f1_score(y_test, y_pred_test, zero_division=0)

    try:
        auc = roc_auc_score(y_test, y_prob_test)
    except ValueError:
        auc = float("nan")

    ce = log_loss(y_test, y_prob_test)

    accuracies.append(acc)
    precisions.append(prec)
    recalls.append(rec)
    f1_scores.append(f1)
    aucs.append(auc)
    ce_losses.append(ce)

    print(
        f"Fold {fold} → Acc={acc:.4f} | Prec={prec:.4f} | "
        f"Rec={rec:.4f} | F1={f1:.4f} | AUC={auc:.4f} | CE Loss={ce:.4f}"
    )

# ============================================================
# Average results
# ============================================================
print("\n=== Average Results Across 20 Folds ===")
print(f"Accuracy:  {np.mean(accuracies):.4f} ± {np.std(accuracies):.4f}")
print(f"Precision: {np.mean(precisions):.4f} ± {np.std(precisions):.4f}")
print(f"Recall:    {np.mean(recalls):.4f} ± {np.std(recalls):.4f}")
print(f"F1-score:  {np.mean(f1_scores):.4f} ± {np.std(f1_scores):.4f}")
print(f"AUC:       {np.nanmean(aucs):.4f} ± {np.nanstd(aucs):.4f}")
print(f"CE Loss:   {np.mean(ce_losses):.4f} ± {np.std(ce_losses):.4f}")

Device: cuda
Images, labels shapes: torch.Size([5856, 3, 224, 224]) torch.Size([5856])


/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Extracted ResNet18 Features: (3583, 512), Labels: (3583,)
Data(x=[3583, 512], edge_index=[2, 1642013])

=== Fold 1 ===
Train Class0: 158, Train Class1: 200
Fold 1 Epoch 1: TotalLoss=0.722918 | Sup=0.712156 | Recon=1.076273 | TrainAcc=0.5587
Fold 1 Epoch 100: TotalLoss=0.134306 | Sup=0.131329 | Recon=0.297649 | TrainAcc=0.9721
Fold 1 Epoch 200: TotalLoss=0.051011 | Sup=0.049027 | Recon=0.198466 | TrainAcc=0.9972
Fold 1 Epoch 300: TotalLoss=0.022719 | Sup=0.020916 | Recon=0.180357 | TrainAcc=1.0000
Fold 1 Epoch 400: TotalLoss=0.013170 | Sup=0.011444 | Recon=0.172607 | TrainAcc=1.0000
Fold 1 Epoch 500: TotalLoss=0.008730 | Sup=0.007068 | Recon=0.166166 | TrainAcc=1.0000
Fold 1 Epoch 600: TotalLoss=0.007580 | Sup=0.005963 | Recon=0.161729 | TrainAcc=1.0000
Fold 1 Epoch 700: TotalLoss=0.004261 | Sup=0.002693 | Recon=0.156804 | TrainAcc=1.0000
Fold 1 Epoch 800: TotalLoss=0.003985 | Sup=0.002452 | Recon=0.153324 | TrainAcc=1.0000
Fold 1 Epoch 900: TotalLoss=0.004269 | Sup=0.002762 | Recon=0.1